## 3.2 Task: Continuous Price Prediction Modeling

In this task, we developed a regression model to predict the continuous price value of an e-commerce order. The goal is to estimate order price using the available transaction features. We used a Multiple Linear Regression approach trained with gradient descent through Scikit-learn's SGDRegressor.

This task includes:
- preparing the regression dataset
- building a preprocessing + regression pipeline
- experimenting with gradient descent parameters
- evaluating the model using MSE and MAE
- applying k-fold cross-validation
- analyzing underfitting and overfitting

In [ ]:
print("3.2 Task: Continuous Price Prediction Modeling")

In [ ]:
# Load data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/cleaned_data.csv")
df.head()

In [ ]:
# Define Features & Target (Regression)

# Regression target
y = df["price"]

# Input features
X = df.drop(columns=["price", "delivery_status", "customer_segment"], errors="ignore")

In [ ]:
# Identify Column Types

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

numerical_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
# Build preprocessing pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# Split the data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


# Build baseline regression model
from sklearn.linear_model import SGDRegressor

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", SGDRegressor(
        loss="squared_error",
        learning_rate="constant",
        eta0=0.01,
        max_iter=1000,
        tol=1e-3,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

# Predict and evaluate baseline model
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Baseline Model Performance")
print("Train MSE:", train_mse)
print("Test MSE :", test_mse)
print("Train MAE:", train_mae)
print("Test MAE :", test_mae)
print("Train R² :", train_r2)
print("Test R²  :", test_r2)



In [ ]:
# Cross-validation

from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_mse_scores = -cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_squared_error"
)

cv_mae_scores = -cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="neg_mean_absolute_error"
)

print("Cross-Validation Results")
print("CV MSE scores:", cv_mse_scores)
print("Average CV MSE:", cv_mse_scores.mean())
print("CV MAE scores:", cv_mae_scores)
print("Average CV MAE:", cv_mae_scores.mean())

In [ ]:
# Experiment with gradient descent settings

experiments = [
    {"eta0": 0.001, "max_iter": 500,  "learning_rate": "constant"},
    {"eta0": 0.001, "max_iter": 1000, "learning_rate": "constant"},
    {"eta0": 0.01,  "max_iter": 1000, "learning_rate": "constant"},
    {"eta0": 0.05,  "max_iter": 1000, "learning_rate": "constant"},
    {"eta0": 0.01,  "max_iter": 2000, "learning_rate": "constant"},
    {"eta0": 0.01,  "max_iter": 1000, "learning_rate": "optimal"},
]

results = []

for exp in experiments:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", SGDRegressor(
            loss="squared_error",
            eta0=exp["eta0"],
            max_iter=exp["max_iter"],
            learning_rate=exp["learning_rate"],
            tol=1e-3,
            random_state=42
        ))
    ])
    
    pipe.fit(X_train, y_train)
    
    train_pred = pipe.predict(X_train)
    test_pred = pipe.predict(X_test)
    
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    cv_mse = -cross_val_score(
        pipe, X_train, y_train,
        cv=cv, scoring="neg_mean_squared_error"
    ).mean()
    
    results.append({
        "learning_rate_type": exp["learning_rate"],
        "eta0": exp["eta0"],
        "epochs_max_iter": exp["max_iter"],
        "train_mse": train_mse,
        "test_mse": test_mse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "cv_mse": cv_mse
    })

    
results_df = pd.DataFrame(results).sort_values(by="test_mse")
results_df

### Note on batch size
Scikit-learn's `SGDRegressor` performs stochastic gradient descent internally and does not provide a direct `batch_size` parameter like TensorFlow or PyTorch. Therefore, in this notebook we experimentally varied the learning rate and the number of training iterations (`max_iter`) to study gradient descent behavior. In the conceptual discussion, batch size is still explained because it is part of the theoretical requirement of the task.

In [ ]:
# Select best model

best_row = results_df.iloc[0]
best_row

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", SGDRegressor(
        loss="squared_error",
        eta0=best_row["eta0"],
        max_iter=int(best_row["epochs_max_iter"]),
        learning_rate=best_row["learning_rate_type"],
        tol=1e-3,
        random_state=42
    ))
])

best_model.fit(X_train, y_train)
best_test_pred = best_model.predict(X_test)

best_test_mse = mean_squared_error(y_test, best_test_pred)
best_test_mae = mean_absolute_error(y_test, best_test_pred)

print("Best Model Test MSE:", best_test_mse)
print("Best Model Test MAE:", best_test_mae)

In [ ]:
# Visualization: actual vs predicted

plt.figure(figsize=(8, 4))
plt.plot(y_test.values[:100], label="Actual", marker="o")
plt.plot(best_test_pred[:100], label="Predicted", marker="x")
plt.title("Actual vs Predicted Prices (First 100 Test Samples)")
plt.xlabel("Sample Index")
plt.ylabel("Price")
plt.legend()
plt.show()

## Effect of Learning Rate, Epochs, and Batch Size

### Learning Rate
The learning rate controls how large each update step is during gradient descent.  
- A very small learning rate makes learning slow but stable.
- A very large learning rate may cause unstable updates or overshooting.
- In our experiments, moderate values gave better test performance than extremely small or large values.

### Epochs
In Scikit-learn's SGDRegressor, `max_iter` represents the maximum number of passes over the training data.
- Too few iterations may lead to underfitting because the model has not learned enough.
- Too many iterations may increase training time and may sometimes lead to overfitting if the model becomes too adapted to the training data.

### Batch Size
Batch size refers to the number of samples processed before updating the model weights.
- Small batch sizes produce noisier but faster updates.
- Large batch sizes produce smoother but slower updates.
- Scikit-learn's SGDRegressor does not directly expose a batch size parameter, so our implementation focuses mainly on learning rate and iteration tuning while discussing batch size conceptually.

## MSE and MAE Interpretation

Mean Squared Error (MSE) measures the average squared difference between actual and predicted values. Because the errors are squared, large errors are penalized more heavily.

Mean Absolute Error (MAE) measures the average absolute difference between actual and predicted values. It is easier to interpret because it shows the average error in the same unit as the target price.

In the AuraCart business context:
- A high MSE means some orders are being predicted very badly, which can harm revenue forecasting.
- A high MAE means the average pricing prediction error is large, which reduces trust in the system.

Large pricing errors may lead to poor financial planning, inaccurate forecasting, and weaker operational decisions.

## Underfitting and Overfitting Analysis

To determine whether the model underfits or overfits, we compared training error, testing error, and cross-validation error.

- If both training and testing errors are high, the model is likely underfitting.
- If training error is low but testing error is much higher, the model is likely overfitting.
- If training, testing, and cross-validation errors are reasonably close, the model is generalizing well.

In our results, the comparison between training MSE, testing MSE, and average cross-validation MSE helps us decide whether the model is stable enough for price prediction.

In [ ]:
# Save final regression pipeline

# import joblib

# joblib.dump(final_regression_pipeline, "../artifacts/final_regression_pipeline.joblib")
# print("Final regression pipeline saved successfully.")


from sklearn.pipeline import Pipeline
import joblib
import os

# Ensure folder exists
os.makedirs("../artifacts", exist_ok=True)

# Create pipeline
final_regression_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

# Save
joblib.dump(final_regression_pipeline, "../artifacts/final_regression_pipeline.joblib")

print("Final regression pipeline saved successfully.")

## 3.3 Task: Multi-class Classification Modeling

In this task, we built a multi-class classification model to predict categorical outcomes in the e-commerce dataset. According to the project requirements, the model should classify transactional events such as `delivery_status` and `customer_segment` using the available input features.

We used Softmax Regression, implemented through Scikit-learn's `LogisticRegression` with `multi_class="multinomial"`. This model outputs probabilities for each class and predicts the class with the highest probability by default.

This task includes:
- preparing classification datasets
- building a preprocessing + classification pipeline
- training a Softmax Regression model
- evaluating classification performance
- analyzing class imbalance
- experimenting with decision thresholds
- interpreting confusion matrices in a business context

In [ ]:
print("Task 3.3 - Multi-class Classification Modeling")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/cleaned_data.csv")
df.head()

In [ ]:
# Define features and target
y_status = df["delivery_status"]

X_status = df.drop(columns=["delivery_status", "customer_segment", "price"], errors="ignore")
X_status.head()

# Check class distribution
status_distribution = y_status.value_counts()
status_percent = y_status.value_counts(normalize=True) * 100

print(status_distribution)
print(status_percent)

plt.figure(figsize=(7, 4))
sns.countplot(x=y_status, order=y_status.value_counts().index)
plt.title("Delivery Status Class Distribution")
plt.xticks(rotation=45)
plt.show()

### Class Imbalance Observation
The `delivery_status` target is imbalanced, with the `Delivered` class appearing much more frequently than `Pending` or `Returned`. This is important because a classifier may achieve high overall accuracy by mostly predicting the majority class, while still performing poorly on rare but business-critical classes such as `Returned`.

In [ ]:
# Identify column types

categorical_cols = X_status.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = [col for col in X_status.columns if col not in categorical_cols]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)


# Build preprocessing pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# Train-test split with stratification
from sklearn.model_selection import train_test_split

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_status,
    y_status,
    test_size=0.2,
    random_state=42,
    stratify=y_status
)

print("Training samples:", X_train_s.shape[0])
print("Testing samples:", X_test_s.shape[0])

# Build Softmax Regression model
from sklearn.linear_model import LogisticRegression

status_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        # multi_class="multinomial",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
# Train the model
status_model.fit(X_train_s, y_train_s)

In [ ]:
# Predictions and probabilities
y_pred_s = status_model.predict(X_test_s)
y_prob_s = status_model.predict_proba(X_test_s)

print("Predicted classes:", status_model.named_steps["classifier"].classes_)
print("Probability matrix shape:", y_prob_s.shape)

In [ ]:
# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy_s = accuracy_score(y_test_s, y_pred_s)
print("Accuracy:", accuracy_s)

print("\nClassification Report:\n")
print(classification_report(y_test_s, y_pred_s))

cm_s = confusion_matrix(y_test_s, y_pred_s, labels=status_model.named_steps["classifier"].classes_)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_s,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=status_model.named_steps["classifier"].classes_,
    yticklabels=status_model.named_steps["classifier"].classes_
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Delivery Status")
plt.show()

In [ ]:
# Cross-validation
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracy = cross_val_score(status_model, X_train_s, y_train_s, cv=cv, scoring="accuracy")
cv_f1_macro = cross_val_score(status_model, X_train_s, y_train_s, cv=cv, scoring="f1_macro")

print("CV Accuracy scores:", cv_accuracy)
print("Average CV Accuracy:", cv_accuracy.mean())

print("CV F1 Macro scores:", cv_f1_macro)
print("Average CV F1 Macro:", cv_f1_macro.mean())

In [ ]:
# Threshold adjustment for rare classes
class_labels = status_model.named_steps["classifier"].classes_
print("Class labels:", class_labels)

# manually adjust the threshold.
returned_index = list(class_labels).index("Returned")
returned_probs = y_prob_s[:, returned_index]

threshold = 0.20
adjusted_pred = []

for i in range(len(y_prob_s)):
    if returned_probs[i] >= threshold:
        adjusted_pred.append("Returned")
    else:
        adjusted_pred.append(class_labels[np.argmax(y_prob_s[i])])

adjusted_pred = np.array(adjusted_pred)

# evaluate
print("Classification Report After Threshold Adjustment:\n")
print(classification_report(y_test_s, adjusted_pred))

cm_adjusted = confusion_matrix(y_test_s, adjusted_pred, labels=class_labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm_adjusted,
    annot=True,
    fmt="d",
    cmap="Oranges",
    xticklabels=class_labels,
    yticklabels=class_labels
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix After Threshold Adjustment")
plt.show()



In [ ]:
# Compare default vs adjusted threshold
from sklearn.metrics import precision_score, recall_score, f1_score

default_precision = precision_score(y_test_s, y_pred_s, labels=["Returned"], average=None, zero_division=0)[0]
default_recall = recall_score(y_test_s, y_pred_s, labels=["Returned"], average=None, zero_division=0)[0]
default_f1 = f1_score(y_test_s, y_pred_s, labels=["Returned"], average=None, zero_division=0)[0]

adjusted_precision = precision_score(y_test_s, adjusted_pred, labels=["Returned"], average=None, zero_division=0)[0]
adjusted_recall = recall_score(y_test_s, adjusted_pred, labels=["Returned"], average=None, zero_division=0)[0]
adjusted_f1 = f1_score(y_test_s, adjusted_pred, labels=["Returned"], average=None, zero_division=0)[0]

comparison_df = pd.DataFrame({
    "Method": ["Default Threshold", "Adjusted Threshold"],
    "Returned Precision": [default_precision, adjusted_precision],
    "Returned Recall": [default_recall, adjusted_recall],
    "Returned F1": [default_f1, adjusted_f1]
})

comparison_df

## Why Linear Regression Is Not Suitable for Classification

Linear regression predicts continuous numerical values, while classification requires assigning an input to one of several discrete categories. If linear regression were used for classification, it could produce outputs outside the valid range of class probabilities, such as negative values or values greater than 1. Therefore, it is not appropriate for categorical prediction tasks such as `delivery_status` or `customer_segment`.

## How Softmax Regression Works

Softmax Regression, also called Multinomial Logistic Regression, is used when the target variable has more than two classes. The model first computes a score for each class. These scores are then passed through the Softmax function, which converts them into probabilities that sum to 1.

For each input:
- the model gives a probability for each class
- the highest probability is normally chosen as the predicted class

This makes the model suitable for multi-class problems such as predicting `Delivered`, `Shipped`, `Pending`, or `Returned`.

## Log-Loss (Categorical Cross-Entropy)

Log-loss measures how different the predicted class probabilities are from the true class labels. A lower log-loss means the predicted probabilities are closer to the correct class.

This is useful because classification is not just about whether the final class is correct, but also about how confident the model is in its predictions. A model that assigns high probability to the wrong class receives a larger penalty.

## Business Meaning of Misclassification Errors

The confusion matrix shows how often the model predicts each class correctly or incorrectly.

In the AuraCart context:
- Predicting a `Returned` order as `Delivered` is risky because the business may fail to identify problematic orders early.
- Predicting `Pending` as `Shipped` may create incorrect expectations in logistics and customer communication.
- Predicting many orders as `Delivered` can inflate accuracy while hiding poor minority-class performance.

Therefore, evaluating only accuracy is not enough. Precision, recall, and F1-score are necessary to understand whether the model performs well for all classes, especially the important rare classes.

                    Predicting customer_segment

In [ ]:
# Define features and target for customer segment
y_segment = df["customer_segment"]

X_segment = df.drop(columns=["customer_segment", "delivery_status", "price"], errors="ignore")
X_segment.head()


# Check class distribution
segment_distribution = y_segment.value_counts()
segment_percent = y_segment.value_counts(normalize=True) * 100

print(segment_distribution)
print(segment_percent)

plt.figure(figsize=(7, 4))
sns.countplot(x=y_segment, order=y_segment.value_counts().index)
plt.title("Customer Segment Class Distribution")
plt.xticks(rotation=45)
plt.show()

# Identify column types
categorical_cols_seg = X_segment.select_dtypes(include=["object"]).columns.tolist()
numerical_cols_seg = [col for col in X_segment.columns if col not in categorical_cols_seg]

print("Categorical columns:", categorical_cols_seg)
print("Numerical columns:", numerical_cols_seg)

# Build preprocessing pipeline
preprocessor_seg = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols_seg),
    ("cat", categorical_transformer, categorical_cols_seg)
])

# Train-test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_segment,
    y_segment,
    test_size=0.2,
    random_state=42,
    stratify=y_segment
)

# Build and train Softmax model
segment_model = Pipeline([
    ("preprocessor", preprocessor_seg),
    ("classifier", LogisticRegression(
        # multi_class="multinomial",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

segment_model.fit(X_train_c, y_train_c)

In [ ]:
# Predict and evaluate
y_pred_c = segment_model.predict(X_test_c)
y_prob_c = segment_model.predict_proba(X_test_c)

accuracy_c = accuracy_score(y_test_c, y_pred_c)
print("Customer Segment Accuracy:", accuracy_c)

print("\nClassification Report:\n")
print(classification_report(y_test_c, y_pred_c))

cm_c = confusion_matrix(y_test_c, y_pred_c, labels=segment_model.named_steps["classifier"].classes_)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_c,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=segment_model.named_steps["classifier"].classes_,
    yticklabels=segment_model.named_steps["classifier"].classes_
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Customer Segment")
plt.show()

In [ ]:
# Cross-validation for customer segment model
cv_accuracy_c = cross_val_score(segment_model, X_train_c, y_train_c, cv=cv, scoring="accuracy")
cv_f1_macro_c = cross_val_score(segment_model, X_train_c, y_train_c, cv=cv, scoring="f1_macro")

print("Customer Segment CV Accuracy:", cv_accuracy_c)
print("Average CV Accuracy:", cv_accuracy_c.mean())

print("Customer Segment CV F1 Macro:", cv_f1_macro_c)
print("Average CV F1 Macro:", cv_f1_macro_c.mean())

In [ ]:
# Threshold adjustment for minority segment
segment_labels = segment_model.named_steps["classifier"].classes_
print("Segment labels:", segment_labels)

minority_class = "New"
minority_index = list(segment_labels).index(minority_class)
minority_probs = y_prob_c[:, minority_index]

threshold = 0.25
adjusted_segment_pred = []

for i in range(len(y_prob_c)):
    if minority_probs[i] >= threshold:
        adjusted_segment_pred.append(minority_class)
    else:
        adjusted_segment_pred.append(segment_labels[np.argmax(y_prob_c[i])])

adjusted_segment_pred = np.array(adjusted_segment_pred)

print("Customer Segment Report After Threshold Adjustment:\n")
print(classification_report(y_test_c, adjusted_segment_pred))

In [ ]:
from sklearn.pipeline import Pipeline
import joblib
import os

# Create artifacts folder
os.makedirs("../artifacts", exist_ok=True)

# Create pipeline (IMPORTANT)
final_classification_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

# Save it
joblib.dump(final_classification_pipeline, "../artifacts/final_classification_pipeline.joblib")

print("Final classification pipeline saved successfully.")